In [18]:
import pandas as pd
import os
import numpy as np
import plotly.graph_objects as go
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [19]:

# Cargar los datos desde el archivo Excel
df = pd.read_csv("../data/outputs/kakebo_merged.csv", sep=";")
df

,MES,MONTO,año
0,MES ENERO,14057968,2025
1,MES FEBRERO,3310580,2025
2,MES MARZO,3072536,2025
3,MES ABRIL,3455498,2025
4,MES MAYO,3385952,2025
5,MES JUNIO,3167049,2025
6,MES JULIO,3337777,2025
7,MES AGOSTO,4465156,2025
8,MES SEPTIEMBRE,5206932,2025
9,MES OCTUBRE,6291127,2025


In [20]:
df.mean

<bound method DataFrame.mean of                MES     MONTO   año
0        MES ENERO  14057968  2025
1      MES FEBRERO   3310580  2025
2        MES MARZO   3072536  2025
3        MES ABRIL   3455498  2025
4         MES MAYO   3385952  2025
5        MES JUNIO   3167049  2025
6        MES JULIO   3337777  2025
7       MES AGOSTO   4465156  2025
8   MES SEPTIEMBRE   5206932  2025
9      MES OCTUBRE   6291127  2025
10   MES NOVIEMBRE   5725800  2025
11   MES DICIEMBRE   6068471  2025
12       MES ENERO   8807436  2026
13     MES FEBRERO   4331947  2026>

In [21]:
df.describe()

,MONTO,año
count,1.400000e+01,14.000000
mean,5.334588e+06,2025.142857
std,2.997040e+06,0.363137
min,3.072536e+06,2025.000000
25%,3.349821e+06,2025.000000
50%,4.398552e+06,2025.000000
75%,5.982803e+06,2025.000000
max,1.405797e+07,2026.000000


In [22]:
# Usar los datos existentes en el notebook para entrenar y predecir
df_hist_actual = df.copy()
df_hist_actual["No MES"] = np.arange(1, len(df_hist_actual) + 1)

X_train = df_hist_actual[["No MES"]]
y_train = df_hist_actual["MONTO"]

modelo_lineal = LinearRegression()
modelo_lineal.fit(X_train, y_train)

# Predicción para el siguiente mes
siguiente_mes_idx = X_train["No MES"].max() + 1
pred_monto = modelo_lineal.predict(pd.DataFrame({"No MES": [siguiente_mes_idx]}))[0]

# Crear tabla con MONTO_REAL, TIPO y año
df_salida = df_hist_actual[["MES", "MONTO", "año"]].copy()
df_salida.rename(columns={"MONTO": "MONTO_REAL"}, inplace=True)
df_salida["TIPO"] = "REAL"

fila_pred = pd.DataFrame({
    "MES": ["SIGUIENTE MES"],
    "MONTO_REAL": [int(round(pred_monto, 0))],
    "año": [df_hist_actual["año"].max()],
    "TIPO": ["PREDICCION"]
})

df_salida = pd.concat([df_salida, fila_pred], ignore_index=True)
df_salida

,MES,MONTO_REAL,año,TIPO
0,MES ENERO,14057968,2025,REAL
1,MES FEBRERO,3310580,2025,REAL
2,MES MARZO,3072536,2025,REAL
3,MES ABRIL,3455498,2025,REAL
4,MES MAYO,3385952,2025,REAL
5,MES JUNIO,3167049,2025,REAL
6,MES JULIO,3337777,2025,REAL
7,MES AGOSTO,4465156,2025,REAL
8,MES SEPTIEMBRE,5206932,2025,REAL
9,MES OCTUBRE,6291127,2025,REAL


In [23]:
df_salida.to_csv("../data/outputs/kakeboo_pred.csv", sep=";", index=False)